# Combined CXR Pipeline — Modal GPU Notebook

## 0. Install & Imports

In [ ]:
%pip install -r requirements.txt

import os
import sys
import pickle
import gc
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from collections import Counter
import torch

# Ensure src modules are discoverable
sys.path.insert(0, os.path.abspath('.'))

## 1. Config & Paths

In [ ]:
from src import config

config.ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Base Path: {config.BASE_PATH}")
print(f"Artifacts Dir: {config.ARTIFACTS_DIR}")
print(f"Images Dir: {config.IMAGES_DIR}")
print(f"Reports File: {config.REPORTS_FILE}")
print(f"Projections File: {config.PROJECTIONS_FILE}")

## 2. Load Data (Reports + Projections)

In [ ]:
from src.data_loader import load_data

df_clean = load_data(config.REPORTS_FILE, config.PROJECTIONS_FILE)
print(f"Clean DataFrame Shape: {df_clean.shape}")

## 3. BioMedCLIP Image Embeddings  [cache-aware]

In [ ]:
from src.embeddings import compute_openclip_image_embeddings
import open_clip

CACHE_PATH = config.ARTIFACTS_DIR / "image_embeddings.pkl"

if CACHE_PATH.exists():
    with open(CACHE_PATH, "rb") as f:
        filename_to_embedding = pickle.load(f)
    print(f"Loaded from cache: {CACHE_PATH}")
else:
    print("Computing BioMedCLIP image embeddings...")
    model, _, preprocess = open_clip.create_model_and_transforms(config.BIOMEDCLIP_MODEL)
    model.to(config.DEVICE)
    model.eval()
    
    unique_filenames = df_clean['filename'].dropna().unique().tolist()
    filename_to_embedding = compute_openclip_image_embeddings(
        filenames=unique_filenames,
        images_dir=config.IMAGES_DIR,
        preprocess=preprocess,
        model=model,
        device=config.DEVICE,
        batch_size=config.OPENCLIP_BATCH_SIZE,
        num_workers=config.NUM_IMAGE_WORKERS
    )
    with open(CACHE_PATH, "wb") as f:
        pickle.dump(filename_to_embedding, f)
    print(f"Saved to cache: {CACHE_PATH}")

# Apply embeddings and clean NaN
df_clean['image_embedding'] = df_clean['filename'].map(filename_to_embedding)
df_clean = df_clean.dropna(subset=['image_embedding']).copy()

## 4. RAD-DINO Image Embeddings    [cache-aware, checkpoint every 10 batches]

In [ ]:
from src.embeddings import compute_hf_vision_embeddings
from transformers import AutoImageProcessor, AutoModel

CACHE_PATH = config.ARTIFACTS_DIR / "raddino_embeddings.pkl"

if CACHE_PATH.exists():
    with open(CACHE_PATH, "rb") as f:
        dino_filename_to_embedding = pickle.load(f)
    print(f"Loaded from cache: {CACHE_PATH}")
else:
    print("Computing RAD-DINO image embeddings...")
    dino_processor = AutoImageProcessor.from_pretrained(config.RADDINO_MODEL)
    dino_model = AutoModel.from_pretrained(config.RADDINO_MODEL).to(config.DEVICE)
    dino_model.eval()
    
    unique_filenames = df_clean['filename'].dropna().unique().tolist()
    dino_filename_to_embedding = compute_hf_vision_embeddings(
        filenames=unique_filenames,
        images_dir=config.IMAGES_DIR,
        processor=dino_processor,
        model=dino_model,
        device=config.DEVICE,
        batch_size=config.RAD_DINO_BATCH_SIZE,
        num_workers=config.NUM_IMAGE_WORKERS,
        checkpoint_path=CACHE_PATH,
        checkpoint_every=10
    )
    with open(CACHE_PATH, "wb") as f:
        pickle.dump(dino_filename_to_embedding, f)
    print(f"Saved to cache: {CACHE_PATH}")

df_clean['raddino_embedding'] = df_clean['filename'].map(dino_filename_to_embedding)
df_clean = df_clean.dropna(subset=['raddino_embedding']).copy()

## 5. Text Embeddings (BioMed-RoBERTa) [cache-aware]

In [ ]:
from src.embeddings import compute_text_embeddings
from transformers import AutoTokenizer, AutoModel

CACHE_PATH = config.ARTIFACTS_DIR / "text_embeddings.pkl"

if CACHE_PATH.exists():
    with open(CACHE_PATH, "rb") as f:
        text_emb = pickle.load(f)
    print(f"Loaded from cache: {CACHE_PATH}")
else:
    print("Computing BioMed-RoBERTa text embeddings...")
    tokenizer = AutoTokenizer.from_pretrained(config.BIOMED_ROBERTA_MODEL)
    text_model = AutoModel.from_pretrained(config.BIOMED_ROBERTA_MODEL).to(config.DEVICE)
    text_model.eval()
    
    text_list = df_clean['full_text'].to_list()
    text_emb = compute_text_embeddings(
        text_list=text_list,
        model=text_model,
        tokenizer=tokenizer,
        device=config.DEVICE,
        batch_size=config.TEXT_BATCH_SIZE
    )
    with open(CACHE_PATH, "wb") as f:
        pickle.dump(text_emb, f)
    print(f"Saved to cache: {CACHE_PATH}")
    
df_clean['text_embedding'] = list(text_emb)

## 6. Train/Test Setup (Full Dataset Replication)

In [ ]:
# The original cxr-blind-retrieval pipeline evaluated *every image* against 
# the entire database without a predefined Train/Test split!
# To rigorously replicate it, we use `df_clean` for both queries & index.

train = df_clean.copy()
test = df_clean.copy()

train_image_embeddings = np.stack(train['image_embedding']).astype('float32')
test_image_embeddings = np.stack(test['image_embedding']).astype('float32')

train_dino_embeddings = np.stack(train['raddino_embedding']).astype('float32')
test_dino_embeddings = np.stack(test['raddino_embedding']).astype('float32')

print(f"Using Full Dataset for Replication: {len(train)} images")

## 7. FAISS Index + Top-10 CLIP Retrieval

In [ ]:
from src.retrieval import build_faiss_index, retrieve_top_k

img_index = build_faiss_index(train_image_embeddings)

# Retrieve top K + 1 (the 1st slot is the exact image itself matched against its own index)
D_clip_full, I_clip_full = retrieve_top_k(img_index, test_image_embeddings, config.RETRIEVAL_K + 1)

# Strip off the self-match at rank 0 (the query retrieving itself distance=1.0)
D_clip = D_clip_full[:, 1:]
I_clip = I_clip_full[:, 1:]

## 8. Consistent vs. Blind Labeling (RAD-DINO threshold=0.60)

In [ ]:
from src.retrieval import label_consistent_blind, mask_to_indices

consistent_mask, blind_mask = label_consistent_blind(
    D_clip=D_clip,
    I_clip=I_clip,
    test_dino_embeddings=test_dino_embeddings,
    train_dino_embeddings=train_dino_embeddings,
    clip_threshold=config.CLIP_THRESHOLD,
    dino_threshold=config.DINO_THRESHOLD
)

test_consistent_neighbors = mask_to_indices(I_clip, consistent_mask)
test_blind_neighbors = mask_to_indices(I_clip, blind_mask)

## 9. RadGraph Entity Extraction   [cache-aware]

In [ ]:
from src.radgraph_utils import load_radgraph, extract_entities

consolidated_rg_cache = config.ARTIFACTS_DIR / "full_dataset_radgraph_entities.pkl"
if consolidated_rg_cache.exists():
    with open(consolidated_rg_cache, "rb") as f:
        full_ents = pickle.load(f)
    print(f"Loaded from cache: {consolidated_rg_cache}")
    train_ground_truth_entities = full_ents
    test_ground_truth_entities = full_ents
else:
    print("Extracting RadGraph entities...")
    radgraph_model = load_radgraph(config.RADGRAPH_MODEL)
    full_ents = extract_entities(
        text_list=df_clean['full_text'].tolist(),
        model=radgraph_model,
        batch_size=config.RADGRAPH_BATCH_SIZE
    )
    with open(consolidated_rg_cache, "wb") as f:
        pickle.dump(full_ents, f)
    print(f"Saved to cache: {consolidated_rg_cache}")
    train_ground_truth_entities = full_ents
    test_ground_truth_entities = full_ents

## 10. Visual Consensus & Deviation Scoring

In [ ]:
from src.analysis import build_consensus

test = build_consensus(
    test_df=test,
    train_df=train,
    I_clip_indices=I_clip,
    test_consistent_neighbors=test_consistent_neighbors,
    test_blind_neighbors=test_blind_neighbors,
    test_entities=test_ground_truth_entities,
    train_entities=train_ground_truth_entities
)
print("Visual Consensus computation complete.")

## 11. CheXpert Pathology Characterization of Blind Pairs

In [ ]:
from src.analysis import connect_blindtype_to_deviation

blind_pair_df = connect_blindtype_to_deviation(
    test_df=test,
    train_df=train,
    df_clean=df_clean,
    test_entities=test_ground_truth_entities,
    train_entities=train_ground_truth_entities
)

print(f"Total blind pairs linked: {len(blind_pair_df)}")
display(blind_pair_df.head())

## 12. Blind Pair Type → RadGraph Deviation Pattern Analysis

In [ ]:
type_counts = blind_pair_df['blind_type'].value_counts()
print("\nBlind Retrieval Types Breakdown:")
print(type_counts)

print("\nTop Deviant RadGraph Entities per Pathology:")
pathology_groups = blind_pair_df.groupby('primary_pathology')
for pathology, group in pathology_groups:
    missing_list = []
    for m_ents in group['missing_entities']:
        missing_list.extend(m_ents)
    
    if len(missing_list) > 0:
        c = Counter(missing_list)
        print(f"\n-- {pathology} --")
        print("Top 5 missing / hallucinated entities:")
        for ent, count in c.most_common(5):
            print(f"  {count}x : {ent}")

## 13. Summary Plots & Results Export

In [ ]:
out_png = config.ARTIFACTS_DIR / "blind_pair_types.png"

fig, ax = plt.subplots(figsize=(7, 5))
types = ['Type 1', 'Type 2', 'Type 3']
counts = [type_counts.get(t, 0) for t in types]

bars = ax.bar(types, counts, color=["#4878CF", "#6ACC65", "#D65F5F"], width=0.5)

for bar, num in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        str(num),
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )

ax.set_title("Blind Retrieval Pairs by Error Type", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of pairs", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig.savefig(out_png, dpi=150)
plt.show()

results_path = config.ARTIFACTS_DIR / "deviation_results.csv"
cols = [
    'uid', 'filename',
    'retrieved_clip_neighbors', 'consistent_neighbors', 'blind_neighbors',
    'radgraph_deviation_full', 'unsupported_entities_full', 'missing_entities_full',
    'radgraph_deviation_consistent', 'unsupported_entities_consistent', 'missing_entities_consistent'
]
test[cols].to_csv(results_path, index=False)
print(f"\nSaved evaluation results to {results_path}")